# 01. 環境設定與測試

本 notebook 將幫助您完成 LangGraph 開發環境的設定與驗證。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 確認 LangGraph 相關套件已正確安裝
- ✅ 設定 OpenAI API Key（可選）
- ✅ 執行第一個 LangGraph 程式
- ✅ 理解 Graph 的基本結構

---

## 📦 LangGraph 生態系

```
┌─────────────────────────────────────────────────────────┐
│                   LangGraph 生態系                       │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌─────────────┐   ┌─────────────┐   ┌─────────────┐  │
│   │  LangGraph  │   │  LangChain  │   │   LangSmith │  │
│   │   (核心)     │   │  (工具整合)  │   │   (監控)    │  │
│   └──────┬──────┘   └──────┬──────┘   └──────┬──────┘  │
│          │                 │                 │         │
│          └─────────────────┴─────────────────┘         │
│                           │                            │
│                    ┌──────▼──────┐                     │
│                    │   您的應用   │                     │
│                    └─────────────┘                     │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

| 套件 | 用途 | 必要性 |
|------|------|--------|
| `langgraph` | 圖結構工作流程引擎 | ✅ 必要 |
| `langchain` | LLM 整合框架 | ✅ 必要 |
| `langchain-openai` | OpenAI 模型支援 | 🔶 可選 |
| `python-dotenv` | 環境變數管理 | 🔶 可選 |

---

## 1.1 檢查套件安裝

首先確認核心套件已正確安裝：

In [1]:
# 檢查核心套件版本
from importlib.metadata import version

packages = ['langgraph', 'langchain', 'langchain-openai', 'langchain-core']

print("📦 已安裝套件：")
print("-" * 40)
for pkg in packages:
    try:
        v = version(pkg)
        print(f"  ✅ {pkg}: {v}")
    except:
        print(f"  ❌ {pkg}: 未安裝")
print("-" * 40)
print("✅ 套件檢查完成！")

📦 已安裝套件：
----------------------------------------
  ✅ langgraph: 1.0.4
  ✅ langchain: 1.1.2
  ✅ langchain-openai: 1.1.0
  ✅ langchain-core: 1.1.1
----------------------------------------
✅ 套件檢查完成！


---

## 1.2 設定 API Key（可選）

如果要使用 OpenAI 模型，請在專案根目錄建立 `.env` 檔案：

```bash
# .env 檔案內容
OPENAI_API_KEY=sk-your-api-key-here
```

> 💡 **提示**：本課程的基礎章節不需要 API Key，您可以先跳過此步驟

In [2]:
import os
from dotenv import load_dotenv

# 載入環境變數
load_dotenv()

# 檢查 API Key 是否設定
api_key = os.getenv("OPENAI_API_KEY")

print("🔑 API Key 狀態：")
print("-" * 40)
if api_key and api_key != "your-api-key-here" and len(api_key) > 10:
    print(f"  ✅ OPENAI_API_KEY 已設定")
    print(f"     前 8 碼: {api_key[:8]}...")
else:
    print("  ⚠️ OPENAI_API_KEY 未設定")
    print("     基礎章節可以正常學習！")
print("-" * 40)

🔑 API Key 狀態：
----------------------------------------
  ✅ OPENAI_API_KEY 已設定
     前 8 碼: sk-proj-...
----------------------------------------


---

## 1.3 LangGraph 核心概念

在測試之前，先了解 LangGraph 的三大核心概念：

```
┌─────────────────────────────────────────────────────────┐
│                   LangGraph 三大元素                     │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌─────────┐         ┌─────────┐         ┌─────────┐  │
│   │  State  │ ──────▶ │  Node   │ ──────▶ │  Edge   │  │
│   │  狀態   │         │  節點   │         │   邊    │  │
│   └─────────┘         └─────────┘         └─────────┘  │
│       │                   │                   │        │
│       ▼                   ▼                   ▼        │
│   共享資料             處理邏輯            流程連接     │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

| 元素 | 說明 | 類比 |
|------|------|------|
| **State** | 在節點間傳遞的共享資料 | 📋 工作單 |
| **Node** | 執行特定任務的處理單元 | 🏭 工作站 |
| **Edge** | 定義節點間的連接關係 | 🛤️ 傳送帶 |

---

## 1.4 第一個 LangGraph 程式

讓我們建立一個最簡單的 Graph：

```
START ──▶ add_one ──▶ END
           │
           ▼
      value + 1
```

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 步驟 1: 定義狀態
class SimpleState(TypedDict):
    """簡單狀態：只有一個 value 欄位"""
    value: int

# 步驟 2: 定義節點函數
def add_one(state: SimpleState) -> dict:
    """將 value 加 1"""
    current = state["value"]
    new_value = current + 1
    print(f"  📍 add_one 節點: {current} → {new_value}")
    return {"value": new_value}

# 步驟 3: 建構圖
graph = StateGraph(SimpleState)
graph.add_node("add_one", add_one)  # 添加節點
graph.add_edge(START, "add_one")     # 入口邊
graph.add_edge("add_one", END)       # 出口邊

# 步驟 4: 編譯
app = graph.compile()

print("✅ Graph 建構完成！")
print("\n📊 執行測試：")
print("-" * 40)

# 步驟 5: 執行
result = app.invoke({"value": 0})

print("-" * 40)
print(f"\n結果: 輸入 0 → 輸出 {result['value']}")
print("\n✅ LangGraph 基本功能測試成功！")

✅ Graph 建構完成！

📊 執行測試：
----------------------------------------
  📍 add_one 節點: 0 → 1
----------------------------------------

結果: 輸入 0 → 輸出 1

✅ LangGraph 基本功能測試成功！


---

## 1.5 視覺化圖結構

LangGraph 可以輸出 Mermaid 格式的圖結構，方便視覺化：

In [4]:
# 顯示圖的 Mermaid 結構
mermaid_code = app.get_graph().draw_mermaid()
print("📊 圖結構 (Mermaid 格式):")
print("=" * 40)
print(mermaid_code)

print("\n💡 提示: 將上述程式碼貼到 https://mermaid.live 或是 https://www.mermaidchart.com/ 可以看到圖形化結果")

📊 圖結構 (Mermaid 格式):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	add_one(add_one)
	__end__([<p>__end__</p>]):::last
	__start__ --> add_one;
	add_one --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


💡 提示: 將上述程式碼貼到 https://mermaid.live 或是 https://www.mermaidchart.com/ 可以看到圖形化結果


---

## 💡 重點回顧

### 建構 LangGraph 的 5 個步驟

```python
# 1️⃣ 定義狀態
class State(TypedDict):
    value: int

# 2️⃣ 定義節點函數
def my_node(state: State) -> dict:
    return {"value": ...}

# 3️⃣ 建構圖
graph = StateGraph(State)
graph.add_node("name", my_node)
graph.add_edge(START, "name")
graph.add_edge("name", END)

# 4️⃣ 編譯
app = graph.compile()

# 5️⃣ 執行
result = app.invoke({...})
```

### 關鍵 API

| API | 用途 |
|-----|------|
| `StateGraph(State)` | 建立狀態圖 |
| `add_node(name, fn)` | 添加節點 |
| `add_edge(from, to)` | 添加邊 |
| `compile()` | 編譯圖 |
| `invoke(state)` | 執行圖 |

---

## 📝 練習題

1. **修改節點**：將 `add_one` 改成 `multiply_two`（乘以 2）
2. **新增欄位**：在 State 中新增 `name: str` 欄位
3. **串聯節點**：建立兩個節點，先加 1 再乘 2

---

🎉 **恭喜！環境設定完成，您可以開始學習 LangGraph 了！**

下一步：[02. 第一個 Graph](02_first_graph.ipynb)